# 10 — Inspect Feather residuals and open a traceable review

**You will:** read the actual Feather arrays, reproduce WRMS and plots, inspect backend
offsets and coherent features, then prepare PLEB's manual review tables without new QC.
Continue in the same PLEB notebook kernel; PyArrow reads the format directly here.
The exporter has already tested Discovery's own reader and writer inside its SIF.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
import pyarrow.feather as feather
export = ROOT/'native-export'
quality = pd.read_csv(export/'reports/residual_quality.csv').set_index('pulsar')
frames = {}
for psr in PULSARS:
    path = export/'feathers'/f'{psr}_combined.feather'
    assert hashlib.sha256(path.read_bytes()).hexdigest()==quality.loc[psr,'feather_sha256']
    arrays = feather.read_table(path).to_pandas()
    # Discovery stores site times, residuals and errors in seconds.
    r = arrays.residuals.to_numpy()
    sigma = arrays.toaerrs.to_numpy()
    assert np.isfinite(r).all() and np.isfinite(sigma).all() and (sigma>0).all()
    centre = np.average(r,weights=1/sigma**2)
    frame = pd.DataFrame(dict(mjd=arrays.stoas/86400,residual_us=(r-centre)*1e6,
        sigma_us=sigma*1e6,frequency_MHz=arrays.freqs,system=arrays.flags_sys))
    m = lab.simulation.metrics(frame)
    assert len(frame)==quality.loc[psr,'n_toas']
    assert abs(m['wrms_us']-quality.loc[psr,'wrms_us'])<1e-9
    frames[psr] = frame
    lab.simulation.plot_residuals(frame,psr+': actual Feather, one global phase zero')
    display(frame.groupby('system').apply(lambda g:pd.Series(lab.simulation.metrics(g))))

## What would make you stop?

Inspect the full vertical range, time and frequency together. Backend means above are
diagnostics only: never subtract a separate mean from each backend to hide a missing JUMP.
Check low/high-frequency offsets, coherent drifts, abrupt changes and isolated deviations
in units of their uncertainty. Small RMS alone can hide excluded good data or a poor model.

Synthetic expectations: each pulsar has about 0.7 µs WRMS after removing the injected
anomalies, no remaining 50 µs features, and both A/B systems represented. The deliberately
broad limits below catch gross exercise failures; they are not release thresholds for
real pulsars. For real data, compare approved reference plots, common-TOA fits, model
coefficients/uncertainties and the exact changed selection, with noise assumptions stated.

In [ ]:
for psr,frame in frames.items():
    assert set(frame.system)=={'A','B'}
    assert lab.simulation.metrics(frame)['wrms_us']<2.0
    assert frame.residual_us.abs().max()<10.0
    w = 1/frame.sigma_us.to_numpy()**2
    z = frame.residual_us.to_numpy()/frame.sigma_us.to_numpy()
    print(psr,'max |residual/error|',float(np.max(np.abs(z))),
          'centred chi-square / N',float(np.sum(z*z)/len(z)))
# No automatic rejection is made from these diagnostic values.

## Prepare PLEB review inputs from the validated export

This command records each physical TIM path, 1-based source line and exact row SHA-256,
together with the verified residual/error arrays in original TIM order. Feather arrays
are sorted; never assume their row indices are physical TIM line numbers.
It does not re-run a detector or fit, and all initial bad/event flags are false.

In [ ]:
review = ROOT/'residual-review'
lab.command([sys.executable,'-m','pleb.qc_review','--from-residual-export',export,'--out',review],
            log=ROOT/'logs/10_prepare_review.log')
from pleb.qc_review import load_qc_frames
review_rows = load_qc_frames(review)
assert len(review_rows)==int(quality.n_toas.sum()) and review_rows.review_id.is_unique
assert not review_rows.bad_point.any() and not review_rows.event_member.any()
display(review_rows[['pulsar','review_id','timfile','toa_line','toa_row_sha256',
                     'tempo2_post_us','tempo2_err_us']].head())
frozen = export/'input/data'
for row in review_rows.itertuples():
    raw = (frozen/row.timfile).read_text().splitlines()[int(row.toa_line)-1]
    assert hashlib.sha256(raw.encode()).hexdigest()==row.toa_row_sha256

## Optional interactive review

Run the command printed below in a host terminal with Streamlit installed in the
environment importing this checkout. Open `http://127.0.0.1:8501`. The notebook server
and review app use separate ports. Select observations, give a reason and reviewer,
and save overrides/reviewed QC; keep both the unchanged input tables and decision history.
This launches the current checkout's UI, including the residual/error unit fix.

If the review finds a real problem, use a **new** apply-only config based on 05, with
only `mark_bad`, comment action and the reviewed output directory. Audit it, fit and
approve a new model if necessary, then freeze a new checkpoint/manifest. Do not edit
this frozen export or expect dataset-build to import UI decisions automatically.

In [ ]:
import shlex
host_root = Path(os.environ['PLEB_LEARNING_HOST_ROOT'])
host_source = Path(os.environ['PLEB_LEARNING_HOST_SOURCE'])
command = ['env','PYTHONPATH='+str(host_source),
    'PLEB_LOG_DIR='+str(host_root/'runtime/review-logs'),
    'MPLCONFIGDIR='+str(host_root/'runtime/review-matplotlib'),
    'python','-m','streamlit','run',str(host_source/'scripts/pleb_qc_review.py'),
    '--server.address','127.0.0.1','--server.port','8501','--server.headless','true','--',
    '--run-dir',str(host_root/'residual-review')]
print(shlex.join(command))

**Your task:** explain why `pqc=good`, an explicit human `keep`, and an unreviewed
automatic keep are different evidence. Follow one plotted point back to its TIM row
and hash. Write the command/config sequence you would use to publish one new rejection.